# Problem 7 - Recitation 3

In [ ]:
# Install if you haven't already. You need this package to open stata datasets.
# install.packages("haven")

# Load the package
library(haven)
library(data.table)


In [ ]:
# Read the .dta file. Set path to the data.
df <- read_dta("Earnings_and_Height/Earnings_and_Height.dta")

In [ ]:
# See how data looks like
head(df)

In [ ]:
# Get a general overview of the variables in df
summary(df)

In [ ]:
# Get column names
colnames(df)

In [ ]:
# Check if there is any missing values in each column
colSums(is.na(df))


### Handy Helpers

| R Function                  | Description                                      |
|-----------------------------|--------------------------------------------------|
| `summary(df)`               | Summary statistics for each column               |
| `str(df)`                   | Structure: types, length, and sample values      |
| `dim(df)`                   | Dimensions: rows and columns                     |
| `nrow(df)`                  | Number of rows                                   |
| `ncol(df)`                  | Number of columns                                |
| `names(df)`                 | Column names                                     |
| `head(df)`                  | First 6 rows                                     |
| `tail(df)`                  | Last 6 rows                                      |
| `colSums(is.na(df))`        | Missing values per column                        |
| `sum(is.na(df))`            | Total number of missing values                   |
| `any(is.na(df))`            | Check if any missing values exist                |
| `glimpse(df)` (from `dplyr`)| Tidy overview of the data structure              |
| `sapply(df, class)`         | Data type of each column                         |


---

### a. Median height in the sample

In [ ]:
median_height <- median(df$height, na.rm = TRUE)
median_height

### b. Average earnings by height groups

In [ ]:
# Split into two groups
shorter <- df[df$height <= 67, ]
taller  <- df[df$height > 67, ]

# i. Average earnings for height <= 67
mean_short <- mean(shorter$earnings, na.rm = TRUE)
mean_short

# ii. Average earnings for height > 67
mean_tall <- mean(taller$earnings, na.rm = TRUE)
mean_tall


In [ ]:
# iii. Difference in means + 95% CI
diff <- mean_tall - mean_short

t_test_result <- t.test(taller$earnings, shorter$earnings)

list(
  mean_earnings_shorter = mean_short,
  mean_earnings_taller = mean_tall,
  difference = diff,
  conf_interval = t_test_result$conf.int
)

**We divided the sample into two groups:**

- Shorter workers: Height ≤ 67 inches
- Taller workers: Height > 67 inches

Average Earnings

- Shorter workers: $44,488.44
- Taller workers: $49,987.88

Difference in Means

- Taller workers earn, on average, $5,499.44 more than shorter workers.

**95% Confidence Interval**

The 95% confidence interval for the difference in average earnings is approximately $4,706.24 to $6,292.64.

**Conclusion**

There is strong evidence that taller workers earn more on average than shorter workers. The entire confidence interval lies above zero, suggesting this difference is statistically significant at the 5% level.

### c. Scatterplot of earnings vs. height

In [ ]:
par(bg = "white")  

plot(df$height, df$earnings,
     xlab = "Height (inches)", ylab = "Earnings",
     main = "Scatterplot of Earnings vs Height",
     pch = 19, col = rgb(0, 0, 1, 0.5))

📌 Explanation: The data description mentions that earnings are reported in bins (e.g., intervals of $2,500 or $5,000), not as exact values. So multiple people may have identical earnings values, leading to horizontal lines in the plot.


### d. Regression: Earnings ~ Height

In [ ]:
model <- lm(earnings ~ height, data = df)
summary(model)

# i. Estimated slope
slope <- coef(model)["height"]
slope

# ii. Predictions
predict_df <- data.frame(height = c(67, 70, 65))
predictions <- predict(model, newdata = predict_df)
names(predictions) <- c("67 inches", "70 inches", "65 inches")

cat(sprintf("Height: %2d inches → Predicted Earnings: $%0.2f\n", 67, predictions[1]))
cat(sprintf("Height: %2d inches → Predicted Earnings: $%0.2f\n", 70, predictions[2]))
cat(sprintf("Height: %2d inches → Predicted Earnings: $%0.2f\n", 65, predictions[3]))


In [ ]:
unique(df$sex)

### f. Female-only regression

In [ ]:
df_female <- subset(df, sex == 0)
model_female <- lm(earnings ~ height, data = df_female)
summary(model_female)

# i. Slope
slope_female <- coef(model_female)["height"]
slope_female

# ii. Predict change in earnings for 1-inch taller than avg
avg_height_female <- mean(df_female$height, na.rm = TRUE)
predict(model_female, newdata = data.frame(height = avg_height_female + 1)) -
  predict(model_female, newdata = data.frame(height = avg_height_female))


### g. Male-only regression

In [ ]:
df_male <- subset(df, sex == 1)
model_male <- lm(earnings ~ height, data = df_male)
summary(model_male)

# i. Slope
slope_male <- coef(model_male)["height"]
slope_male

# ii. 1-inch taller than avg male
avg_height_male <- mean(df_male$height, na.rm = TRUE)
predict(model_male, newdata = data.frame(height = avg_height_male + 1)) -
  predict(model_male, newdata = data.frame(height = avg_height_male))


```r
# Calculate average male height
avg_height_male <- mean(df_male$height, na.rm = TRUE)

# Predict earnings for a man who is 1 inch taller than average
pred_1inch_taller <- predict(model_male, newdata = data.frame(height = avg_height_male + 1))
pred_avg_height   <- predict(model_male, newdata = data.frame(height = avg_height_male))

# Difference in predicted earnings
diff_pred <- pred_1inch_taller - pred_avg_height

# Print result
cat(sprintf("A male worker who is 1 inch taller than the average earns $%.2f more (according to the model).\n", diff_pred))
```
📌 Were all these steps necessary to answer the question?

### h. Is height uncorrelated with other factors that cause earnings?

#### No R code needed; write-up:

- In theory, height might be correlated with other variables like education, confidence, or even discrimination.
- These unobserved factors could also affect earnings, violating the assumption that $E[u_i | Height]$ = 0.
- If this assumption fails, our regression may suffer from omitted variable bias.